# SLTFM · E2: Base-model comparison on Sri Lankan Tamil

Which base model models **Sri Lankan Tamil** best before any training of ours?

| Model | Why it is here |
|---|---|
| `meta-llama/Meta-Llama-3-8B` | Our current base model |
| `ctnlpr/Llama-3-SLTFM-8B-CPT-LoRA` (checkpoint-1100) | Reference: our CPT model |
| `Qwen/Qwen3.5-4B-Base`, `Qwen/Qwen3.5-9B-Base` | 2.8× more efficient tokenizer for Tamil; 9B ≈ same size as Llama 3 8B |
| `murai-labs/agaram-2.5b` | Tamil-first model trained from scratch; 6.5× more efficient tokenizer |

**Measured** on the 13 validation books (≈1.0M characters, never used in training):
1. **Bits per character (BPC)**: lower is better; fair across different tokenizers. 95% confidence intervals, and the paired difference to Llama 3.
2. The **10 Sri Lankan Tamil questions** and **6 continuations** from the first evaluation notebook.
3. **GPU memory and speed** on a T4.

All models are loaded the same way: 4-bit NF4, fp16 compute (as in our CPT run). Each model runs as a separate process, one model per GPU at a time, so a failure in one model does not stop the others.

### Setup
- **Accelerator:** GPU T4 ×2 · **Internet:** on
- **Secret** `HF_TOKEN` (Add-ons → Secrets): a token from an account that accepted the Llama 3 licence and can read `ctnlpr`

### How to run
1. **Quick test first** (`RUN_QUICK = True`, `RUN_FULL = False`): Run All. Every model loads and scores 20,000 characters. About 30–60 minutes, mostly downloads. Check the log summary for errors.
2. **Full run** (`RUN_FULL = True`): *Save Version → Save & Run All (Commit)*, so it keeps running after you close the browser. Expected a few hours (under Kaggle's 12-hour limit).
3. Download `e2_results.md` and `e2_summary.json` from the Output tab.


In [ ]:
# 1 · Packages (the evaluation runs in separate processes, so no kernel restart is needed)
!pip install -q -U transformers peft accelerate bitsandbytes datasets
# Optional speed-up for Qwen 3.5's linear-attention layers. If this fails, ignore it: the models still run, only slower.
!pip install -q flash-linear-attention || echo "flash-linear-attention not installed (optional)"
!python -c "import transformers, peft, torch, bitsandbytes; print('transformers', transformers.__version__, '| peft', peft.__version__, '| torch', torch.__version__, '| bitsandbytes', bitsandbytes.__version__)"
!nvidia-smi -L
!df -h / | tail -1


In [ ]:
# 2 · Settings
RUN_QUICK   = True       # every model: load + score 20,000 characters, no generation
RUN_FULL    = False      # all validation text + questions + continuations
QUICK_CHARS = 20_000

# models per GPU (balanced by expected time); with one GPU everything runs on GPU 0
QUEUES = [["llama3-8b", "llama3-8b-sltfm-cpt"],
          ["qwen3.5-4b", "agaram-2.5b", "qwen3.5-9b"]]

import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    WORK = "/kaggle/working"
except Exception:
    WORK = "."
assert os.environ.get("HF_TOKEN"), "Add a secret named HF_TOKEN"
print("output folder:", WORK)


In [ ]:
%%writefile e2_eval.py
"""E2 · Base-model comparison on Sri Lankan Tamil (SLTFM).

Scores one model on the validation books of ctnlpr/sri-lankan-tamil-corpus:
  * bits per character (BPC), tokenizer-independent, per 2,000-character segment
    (segments are the same for every model, so models can be compared pair by pair)
  * the 10 Sri Lankan Tamil questions and 6 continuations from the first eval notebook
  * memory and speed on the GPU

Usage:  CUDA_VISIBLE_DEVICES=0 python e2_eval.py --model qwen3.5-4b --out_dir results [--max_chars 20000] [--no_gen]
"""
import argparse, json, math, os, time

import numpy as np
import torch
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from packaging import version
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODELS = {
    "llama3-8b": {"repo": "meta-llama/Meta-Llama-3-8B"},
    "llama3-8b-sltfm-cpt": {"repo": "meta-llama/Meta-Llama-3-8B",
                            "adapter": "ctnlpr/Llama-3-SLTFM-8B-CPT-LoRA", "subfolder": "checkpoint-1100"},
    "qwen3.5-4b": {"repo": "Qwen/Qwen3.5-4B-Base"},
    "qwen3.5-9b": {"repo": "Qwen/Qwen3.5-9B-Base"},
    "agaram-2.5b": {"repo": "murai-labs/agaram-2.5b"},
}
DATASET_ID = "ctnlpr/sri-lankan-tamil-corpus"

SHOT = "கேள்வி: இலங்கையின் வடக்கே உள்ள முக்கிய தமிழ் நகரம் எது?\nபதில்: யாழ்ப்பாணம்.\n\n"
QUESTIONS = [
    ("General", "திருக்குறளை எழுதியவர் யார்?", "திருவள்ளுவர்"),
    ("General", "யாழ்ப்பாணம் இலங்கையின் எந்த மாகாணத்தில் உள்ளது?", "வட மாகாணம்"),
    ("General", "இலங்கையின் வணிகத் தலைநகரம் எது?", "கொழும்பு"),
    ("General", "நல்லூர் கந்தசுவாமி கோவில் எந்த நகரில் அமைந்துள்ளது?", "யாழ்ப்பாணம் (நல்லூர்)"),
    ("Corpus-grounded", "சித்த மருத்துவத்தில் கூறப்படும் மூன்று தோஷங்கள் யாவை?", "வாதம், பித்தம், கபம்"),
    ("Corpus-grounded", "திரிகடுகம் என்பது எந்த மூன்று பொருட்களின் கூட்டு?", "சுக்கு, மிளகு, திப்பிலி"),
    ("Corpus-grounded", "அகத்தியர் இயற்றியதாகக் கூறப்படும் இலக்கண நூல் எது?", "அகத்தியம்"),
    ("Corpus-grounded", "திருக்குறளில் மருத்துவம் பற்றிக் கூறும் அதிகாரத்தின் பெயர் என்ன?", "மருந்து (95ஆம் அதிகாரம்)"),
    ("Sri Lankan usage", "இலங்கைத் தமிழில் 'கதைத்தல்' என்ற சொல்லின் பொருள் என்ன?", "பேசுதல்"),
    ("Sri Lankan usage", "யாழ்ப்பாணத்தில் பனை மரத்திலிருந்து பெறப்படும் இனிப்புப் பொருள் எது?", "பனங்கட்டி / கருப்பட்டி"),
]
CONTINUATIONS = [
    "யாழ்ப்பாணம் இலங்கையின் வடக்கே அமைந்துள்ள",
    "ஈழத்துத் தமிழ் இலக்கிய வரலாற்றில்",
    "சித்த மருத்துவம் என்பது",
    "இலங்கையில் கூட்டுறவு இயக்கம்",
    "மட்டக்களப்பு மாவட்டத்தின் மக்கள்",
    "பனை மரம் யாழ்ப்பாண மக்களின் வாழ்வில்",
]


def load(spec, quant, token):
    tok = AutoTokenizer.from_pretrained(spec["repo"], token=token)
    assert tok.is_fast, "a fast tokenizer is needed for character offsets"
    dtype_key = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"
    kw = {"device_map": {"": 0}, "token": token, dtype_key: torch.float32 if quant == "fp32" else torch.float16}
    if quant == "4bit":
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16)
    try:
        model, how = AutoModelForCausalLM.from_pretrained(spec["repo"], **kw), "AutoModelForCausalLM"
    except Exception as e:  # Qwen 3.5 is packaged as an image-text model
        print("AutoModelForCausalLM failed, trying AutoModelForImageTextToText:", repr(e)[:300], flush=True)
        from transformers import AutoModelForImageTextToText
        model, how = AutoModelForImageTextToText.from_pretrained(spec["repo"], **kw), "AutoModelForImageTextToText"
    if spec.get("adapter"):
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, spec["adapter"], subfolder=spec["subfolder"], token=token)
        how += " + LoRA adapter"
    model.eval()
    return tok, model, how


def device_of(model):
    return next(model.parameters()).device


@torch.no_grad()
def token_nll(model, seq, window, stride):
    """nll[p] = -ln P(seq[p] | seq[:p]) for p >= 1, using overlapping windows (every token scored once)."""
    n = len(seq)
    nll = np.full(n, np.nan)
    prev_end, begin = 1, 0
    while True:
        end = min(begin + window, n)
        x = torch.tensor([seq[begin:end]], device=device_of(model))
        logits = model(input_ids=x).logits[0]
        tgt = x[0, 1:]
        first = max(prev_end, begin + 1)          # first absolute position not scored yet
        rel = first - begin - 1                   # its index in logits
        vals = [F.cross_entropy(logits[a:min(a + 512, end - begin - 1)].float(),
                                tgt[a:min(a + 512, end - begin - 1)], reduction="none")
                for a in range(rel, end - begin - 1, 512)]
        if vals:
            nll[first:end] = torch.cat(vals).cpu().numpy()
        prev_end = end
        if end == n:
            return nll
        begin += min(stride, window - 1)          # keep one token of overlap so no position is skipped


@torch.no_grad()
def generate(tok, model, prompt, max_new_tokens, stop_newline=False):
    x = tok(prompt, return_tensors="pt").to(device_of(model))
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    y = model.generate(**x, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.15,
                       pad_token_id=pad)
    text = tok.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip().split("\n")[0].strip() if stop_newline else text.strip()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True, choices=list(MODELS))
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--quant", default="4bit", choices=["4bit", "fp16", "fp32"])
    ap.add_argument("--max_chars", type=int, default=0, help="0 = all validation text")
    ap.add_argument("--window", type=int, default=2048)
    ap.add_argument("--stride", type=int, default=1024)
    ap.add_argument("--seg_chars", type=int, default=2000)
    ap.add_argument("--no_gen", action="store_true")
    a = ap.parse_args()
    token = os.environ.get("HF_TOKEN")
    spec = MODELS[a.model]
    os.makedirs(a.out_dir, exist_ok=True)

    val = load_dataset(DATASET_ID, "documents", split="validation", token=token)
    docs, budget = [], a.max_chars or 10**12
    for t in val["text"]:
        if budget <= 0:
            break
        docs.append(t[:budget])
        budget -= len(docs[-1])

    t0 = time.time()
    tok, model, how = load(spec, a.quant, token)
    load_s = time.time() - t0
    print(f"[{a.model}] loaded with {how} in {load_s:.0f}s", flush=True)
    start_id = tok.bos_token_id if tok.bos_token_id is not None else tok.eos_token_id

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t1 = time.time()
    seg_rows, n_tok = [], 0
    for d, text in enumerate(docs):
        enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
        ids, offs = enc["input_ids"], enc["offset_mapping"]
        seq = [start_id] + ids
        nll = token_nll(model, seq, a.window, a.stride)[1:]
        if np.isnan(nll).any():
            raise RuntimeError(f"NaN loss in document {d} (try --quant fp32 for small models)")
        seg = np.array([o[0] for o in offs]) // a.seg_chars
        n_seg = math.ceil(len(text) / a.seg_chars)
        bits = np.bincount(seg, weights=nll, minlength=n_seg) / math.log(2)
        for k in range(n_seg):
            chars = min(len(text), (k + 1) * a.seg_chars) - k * a.seg_chars
            seg_rows.append({"doc": d, "seg": k, "bits": float(bits[k]), "chars": chars})
        n_tok += len(ids)
        print(f"[{a.model}] doc {d + 1}/{len(docs)}: {len(text):,} chars, {len(ids):,} tokens, "
              f"running BPC {sum(r['bits'] for r in seg_rows) / sum(r['chars'] for r in seg_rows):.4f} "
              f"[{time.time() - t1:.0f}s]", flush=True)
    score_s = time.time() - t1
    n_chars = sum(r["chars"] for r in seg_rows)
    bpc = sum(r["bits"] for r in seg_rows) / n_chars
    res = {
        "model": a.model, "repo": spec["repo"], "adapter": spec.get("adapter"), "loaded_with": how,
        "quant": a.quant, "window": a.window, "stride": a.stride, "seg_chars": a.seg_chars,
        "n_docs": len(docs), "n_chars": n_chars, "n_tokens": n_tok, "chars_per_token": n_chars / n_tok,
        "bpc": bpc, "peak_mem_gb": torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0,
        "load_s": load_s, "score_s": score_s, "scored_tokens_per_s": n_tok / score_s,
        "chars_per_s": n_chars / score_s,
        "versions": {"transformers": transformers.__version__, "torch": torch.__version__},
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "segments": seg_rows,
    }
    print(f"[{a.model}] BPC {bpc:.4f} on {n_chars:,} chars | {n_chars / n_tok:.2f} chars/token | "
          f"peak {res['peak_mem_gb']:.1f} GB | {res['chars_per_s']:.0f} chars/s", flush=True)

    if not a.no_gen:
        cpt = n_chars / n_tok
        qa_tokens, cont_tokens = int(150 / cpt) + 8, min(600, int(500 / cpt) + 8)
        res["questions"] = []
        for cat, q, exp in QUESTIONS:
            ans = generate(tok, model, SHOT + f"கேள்வி: {q}\nபதில்:", qa_tokens, stop_newline=True)
            res["questions"].append({"category": cat, "question": q, "expected": exp, "answer": ans})
            print(f"  Q: {q}\n  A: {ans}", flush=True)
        res["continuations"] = [{"prompt": p, "text": generate(tok, model, p, cont_tokens)} for p in CONTINUATIONS]

    path = os.path.join(a.out_dir, f"{a.model}.json")
    json.dump(res, open(path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print("saved", path, flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile summarize_e2.py
import glob, json, os
import numpy as np

ORDER = ["llama3-8b", "llama3-8b-sltfm-cpt", "qwen3.5-4b", "qwen3.5-9b", "agaram-2.5b"]


def summarize(res_dir, ref="llama3-8b", n_boot=2000, seed=0):
    """BPC per model with 95% bootstrap CIs over 2,000-char segments, plus the paired difference to `ref`."""
    runs = {}
    for p in glob.glob(os.path.join(res_dir, "*.json")):
        r = json.load(open(p, encoding="utf-8"))
        runs[r["model"]] = r
    if not runs:
        raise SystemExit(f"no results in {res_dir}")
    seg = {m: {(s["doc"], s["seg"]): (s["bits"], s["chars"]) for s in r["segments"]} for m, r in runs.items()}
    keys = sorted(set.intersection(*(set(d) for d in seg.values())))   # segments every model scored
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(keys), size=(n_boot, len(keys)))
    boot, rows = {}, []
    for m in sorted(runs, key=lambda x: ORDER.index(x) if x in ORDER else 99):
        bits = np.array([seg[m][k][0] for k in keys]); chars = np.array([seg[m][k][1] for k in keys])
        boot[m] = bits[idx].sum(1) / chars[idx].sum(1)
        r = runs[m]
        rows.append({"model": m, "bpc": bits.sum() / chars.sum(),
                     "ci_low": float(np.percentile(boot[m], 2.5)), "ci_high": float(np.percentile(boot[m], 97.5)),
                     "chars_per_token": r["chars_per_token"], "peak_mem_gb": r["peak_mem_gb"],
                     "chars_per_s": r["chars_per_s"], "quant": r["quant"], "loaded_with": r["loaded_with"]})
    for row in rows:
        m = row["model"]
        if ref in boot and m != ref:
            d = boot[m] - boot[ref]
            row["diff_vs_ref"] = row["bpc"] - next(x["bpc"] for x in rows if x["model"] == ref)
            row["diff_ci"] = (float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)))
    n_chars = sum(seg[next(iter(seg))][k][1] for k in keys)
    return rows, runs, n_chars, len(keys)


def to_markdown(rows, runs, n_chars, n_seg, ref="llama3-8b"):
    def cell(s):
        return str(s).replace("|", "\\|").replace("\n", " ")
    L = ["# E2 · Base-model comparison on Sri Lankan Tamil", "",
         f"Validation books of `ctnlpr/sri-lankan-tamil-corpus`: {n_chars:,} characters, {n_seg} segments of "
         f"2,000 characters. 95% confidence intervals by bootstrap over segments; the difference to {ref} is paired.", "",
         f"| Model | BPC ↓ | 95% CI | vs {ref} (95% CI) | Chars/token | Peak GPU GB | Chars/s | Loading |",
         "|---|---|---|---|---|---|---|---|"]
    for r in rows:
        diff = (f"{r['diff_vs_ref']:+.3f} ({r['diff_ci'][0]:+.3f} to {r['diff_ci'][1]:+.3f})"
                if "diff_vs_ref" in r else "—")
        L.append(f"| {r['model']} | **{r['bpc']:.3f}** | {r['ci_low']:.3f}–{r['ci_high']:.3f} | {diff} | "
                 f"{r['chars_per_token']:.2f} | {r['peak_mem_gb']:.1f} | {r['chars_per_s']:.0f} | {r['quant']}, {r['loaded_with']} |")
    with_q = [m["model"] for m in rows if "questions" in runs[m["model"]]]
    if with_q:
        L += ["", "## Sri Lankan Tamil questions (one-shot prompt, greedy)", "",
              "| Question | Expected | " + " | ".join(with_q) + " |", "|---|---|" + "---|" * len(with_q)]
        for i, q in enumerate(runs[with_q[0]]["questions"]):
            L.append(f"| {cell(q['question'])} | {cell(q['expected'])} | " +
                     " | ".join(cell(runs[m]["questions"][i]["answer"]) for m in with_q) + " |")
        L += ["", "## Continuations", ""]
        for i, c in enumerate(runs[with_q[0]]["continuations"]):
            L += [f"**Prompt:** {c['prompt']}", ""]
            L += [f"- *{m}:* {cell(runs[m]['continuations'][i]['text'])}" for m in with_q]
            L.append("")
    return "\n".join(L)


In [ ]:
# 3 · Runner: one process per model, one queue per GPU, logs in <out>/logs
import shutil, subprocess, threading, time, torch
from huggingface_hub import constants as hf_constants

REPOS = {"llama3-8b": "meta-llama/Meta-Llama-3-8B", "llama3-8b-sltfm-cpt": "meta-llama/Meta-Llama-3-8B",
         "qwen3.5-4b": "Qwen/Qwen3.5-4B-Base", "qwen3.5-9b": "Qwen/Qwen3.5-9B-Base",
         "agaram-2.5b": "murai-labs/agaram-2.5b"}

def run_all(out_dir, extra_args):
    os.makedirs(f"{out_dir}/logs", exist_ok=True)
    queues = QUEUES if torch.cuda.device_count() >= 2 else [sum(QUEUES, [])]
    pending = {m for q in queues for m in q}
    lock, status = threading.Lock(), {}

    def worker(gpu, names):
        for name in names:
            t0 = time.time()
            with open(f"{out_dir}/logs/{name}.log", "w") as log:
                rc = subprocess.run(["python", "e2_eval.py", "--model", name, "--out_dir", out_dir] + extra_args,
                                    env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)),
                                    stdout=log, stderr=subprocess.STDOUT).returncode
            with lock:
                status[name] = ("ok" if rc == 0 else f"FAILED (exit {rc})", time.time() - t0)
                pending.discard(name)
                repo = REPOS[name]
                if not any(REPOS[m] == repo for m in pending):   # free disk: nobody needs this model any more
                    shutil.rmtree(os.path.join(hf_constants.HF_HUB_CACHE, "models--" + repo.replace("/", "--")),
                                  ignore_errors=True)
            print(f"GPU {gpu} · {name}: {status[name][0]} in {status[name][1] / 60:.0f} min", flush=True)

    threads = [threading.Thread(target=worker, args=(g, q)) for g, q in enumerate(queues)]
    [t.start() for t in threads]; [t.join() for t in threads]
    for name in status:                       # last lines of every log
        lines = open(f"{out_dir}/logs/{name}.log", encoding="utf-8", errors="replace").read().strip().splitlines()
        print(f"\n==== {name}: {status[name][0]} ====")
        print("\n".join(lines[-6:]))
    return status


In [ ]:
# 4 · Quick test
if RUN_QUICK:
    quick_status = run_all(f"{WORK}/e2_quick", ["--max_chars", str(QUICK_CHARS), "--no_gen"])


In [ ]:
# 5 · Full run (set RUN_FULL = True, then Save Version → Save & Run All)
if RUN_FULL:
    full_status = run_all(f"{WORK}/e2_full", [])


In [ ]:
# 6 · Results table (uses the full run if it exists, otherwise the quick test)
import glob, json
from summarize_e2 import summarize, to_markdown

res_dir = f"{WORK}/e2_full" if glob.glob(f"{WORK}/e2_full/*.json") else f"{WORK}/e2_quick"
rows, runs, n_chars, n_seg = summarize(res_dir)
md = to_markdown(rows, runs, n_chars, n_seg)
open(f"{WORK}/e2_results.md", "w", encoding="utf-8").write(md)
json.dump({"source": res_dir, "n_chars": n_chars, "n_segments": n_seg, "models": rows},
          open(f"{WORK}/e2_summary.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("from", res_dir)
print(md)


### How to read the results
- **BPC** is the main number: lower means the model predicts Sri Lankan Tamil text better. It is fair across tokenizers.
- **vs llama3-8b**: if the whole 95% interval is below zero, the model is reliably better than Llama 3 on this text; if it crosses zero, the difference is not clear.
- The quick test (20,000 characters) is only for checking that everything runs; use the full run for decisions.
- **Caveats:** all models are scored in 4-bit; windows are 2,048 tokens, so models with more efficient tokenizers see more characters of context; the CPT model was trained on books from the same collection (different books), so it is a reference, not a fair competitor.

**Decision rule agreed before the run** (fill in with Anna): e.g. switch to Qwen 3.5 if its BPC is equal or better than Llama 3's and it trains on a T4; consider Agaram as a second, low-cost track if its BPC is clearly best, after checking its answers to the knowledge questions.
